# Classifiers

> This module contains code to build a text classification model

In [ ]:
#| default_exp models.classifiers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import torch
from transformers.models.roberta.configuration_roberta import RobertaConfig
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import AutoConfig
from that_nlp_library.utils import *

In [ ]:
#| export
def model_init_classification(
                              model_class, # Model's class object, e.g. RobertaHiddenStateConcatForSequenceClassification
                              cpoint_path, # Either model string name on HuggingFace, or the path to model checkpoint
                              output_hidden_states:bool, # To whether output the model hidden states or not. Useful when you try to build a custom classification head 
                              seed=42, # Random seed
                              model_kwargs={} # Keyword arguments for model
                             ):
    """To initialize a classification model, either from an existing HuggingFace model or custom architecture
    
    Can be used for binary, multi-class single-head, multi-class "two-head", and multi-label clasisifcation
    """    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = AutoConfig.from_pretrained(
        cpoint_path,
        output_hidden_states=output_hidden_states,
    )
    seed_everything(seed)
    model = model_class.from_pretrained(cpoint_path,config=config,**model_kwargs).to(device)
    return model

In [ ]:
show_doc(model_init_classification)

---

[source](https://github.com/anhquan0412/that-nlp-library/blob/main/that_nlp_library/models/classifiers.py#L19){target="_blank" style="float:right; font-size:smaller"}

### model_init_classification

>      model_init_classification (model_class, cpoint_path,
>                                 output_hidden_states:bool, seed=42,
>                                 model_kwargs={})

To initialize a classification model, either from an existing HuggingFace model or custom architecture

Can be used for binary, multi-class single-head, multi-class "two-head", and multi-label clasisifcation

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model_class |  |  | Model's class object, e.g. RobertaHiddenStateConcatForSequenceClassification |
| cpoint_path |  |  | Either model string name on HuggingFace, or the path to model checkpoint |
| output_hidden_states | bool |  | To whether output the model hidden states or not. Useful when you try to build a custom classification head |
| seed | int | 42 | Random seed |
| model_kwargs | dict | {} | Keyword arguments for model |

In [ ]:
#| export
def compute_metrics_classification(pred, # An EvalPrediction object from HuggingFace (which is a named tuple with ```predictions``` and ```label_ids``` attributes)
                                             metric_funcs=[], # A list of metric functions to evaluate
                                             head_sizes=[], # Class size for each head,
                                             label_names=[], # Names of the label (dependent variable) columns
                                             is_multilabel=False, # Whether this is a multilabel classification
                                             multilabel_thres=0.5 # Threshold for multilabel (>= threshold is positive)
                                            ):
    """
    Return a dictionary of metric name and its values. Can handle both multiclass and multilabel    
    
    Reference: https://github.com/huggingface/transformers/blob/dbc12269ed5546b2da9236b9f1078b95b6a4d3d5/src/transformers/trainer_utils.py#LL100C22-L100C22
    """
    assert len(head_sizes)==len(label_names)
    labels = pred.label_ids 
    if isinstance(pred.predictions,tuple):
        preds = pred.predictions[0]
    else:
        preds = pred.predictions
    results={}
    metric_funcs = val2iterable(metric_funcs)
    
    for i,(_size,_name) in enumerate(zip(head_sizes,label_names)):
        start= 0 if i==0 else start+head_sizes[i-1]
        end = start + _size
        _pred = preds[:,start:end]
        if is_multilabel:
            # sigmoid and threshold
            _pred = (sigmoid(_pred)>=multilabel_thres).astype(int)
        else:
            _pred = _pred.argmax(-1)
        _label = labels[:,i] if len(head_sizes)>1 else labels
        for m_func in metric_funcs:
            m_name = callable_name(m_func)
            results[f'{m_name}_{_name}']=m_func(_label,_pred)
    return results

In [ ]:
show_doc(compute_metrics_classification)

---

[source](https://github.com/anhquan0412/that-nlp-library/blob/main/that_nlp_library/models/classifiers.py#L40){target="_blank" style="float:right; font-size:smaller"}

### compute_metrics_classification

>      compute_metrics_classification (pred, metric_funcs=[], head_sizes=[],
>                                      label_names=[], is_multilabel=False,
>                                      multilabel_thres=0.5)

Return a dictionary of metric name and its values. Can handle both multiclass and multilabel    

Reference: https://github.com/huggingface/transformers/blob/dbc12269ed5546b2da9236b9f1078b95b6a4d3d5/src/transformers/trainer_utils.py#LL100C22-L100C22

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| pred |  |  | An EvalPrediction object from HuggingFace (which is a named tuple with ```predictions``` and ```label_ids``` attributes) |
| metric_funcs | list | [] | A list of metric functions to evaluate |
| head_sizes | list | [] | Class size for each head, |
| label_names | list | [] | Names of the label (dependent variable) columns |
| is_multilabel | bool | False | Whether this is a multilabel classification |
| multilabel_thres | float | 0.5 | Threshold for multilabel (>= threshold is positive) |

In [ ]:
#| export
def loss_for_classification(logits, # output of the last linear layer, before any softmax/sigmoid. Size: (bs,class_size)
                            labels, # determined by your datasetdict. Size: (bs,number_of_head)
                            is_multilabel=False, # Whether this is a multilabel classification
                            is_multihead=False, # Whether this is a multihead (multi-level) classification
                            head_sizes=[], # class size for each head
                            head_weights=[], # loss weight for each head
                           ):
    """
    The general loss function for classification
    
    - If is_multilabel is ```False``` and is_multihead is ```False```: One-Head Classification, e.g. You predict 1 out of n class
    
    - If is_multilabel is ```False``` and is_multihead is ```True```: Multi-Head Classification, e.g. You predict 1 out of n classes at Level 1, 
    and 1 out of m classes at Level 2
    
    - If is_multilabel is ```True``` and is_multihead is ```False```: One-Head Multi-Label Classification, e.g. You predict x out of n class (x>=1)
    
    - If is_multilabel is ```True``` and is_multihead is ```True```: Not supported!
    
    """
    if is_multilabel and is_multihead: raise ValueError('Multi-Label and Multi-Head problem is not supported')
    head_sizes = val2iterable(head_sizes)
    loss=0
    if not is_multilabel:
        if not is_multihead:
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, head_sizes[0]), labels.view(-1))
        else:
            assert len(head_sizes)==len(head_weights),"For MultiHead, make sure len of head_sizes and head_weights equal"
            for i,(_size,_weight) in enumerate(zip(head_sizes,head_weights)):
                start= 0 if i==0 else start+head_sizes[i-1]
                end = start + _size
                loss_fct = torch.nn.CrossEntropyLoss()
                loss = loss + _weight*loss_fct(logits[:,start:end].view(-1,_size),
                                               labels[:,i].view(-1))
    else:
        if not is_multihead:
            loss_fct = torch.nn.BCEWithLogitsLoss()
#             label_1hot = torch.nn.functional.one_hot(labels.view(-1),num_classes=head_sizes[0])
            loss = loss_fct(logits,
                            labels.float())
        else:
            raise ValueError('Multi-Head with Multi-Label classification is not supported. Have you lost your mind?')
#             assert len(head_sizes)==len(head_weights),"For MultiHead, make sure len of head_sizes and head_weights equal"
#             for i,(_size,_weight) in enumerate(zip(head_sizes,head_weights)):
#                 start= 0 if i==0 else start+head_sizes[i-1]
#                 end = start + _size
#                 loss_fct = torch.nn.BCEWithLogitsLoss()
#                 loss = loss + _weight*loss_fct(logits[:,start:end].view(-1,_size),
#                                                torch.nn.functional.one_hot(labels[:,i].view(-1),num_classes=_size).float()
#                                               )
            
    return loss

In [ ]:
show_doc(loss_for_classification)

---

[source](https://github.com/anhquan0412/that-nlp-library/blob/main/that_nlp_library/models/classifiers.py#L77){target="_blank" style="float:right; font-size:smaller"}

### loss_for_classification

>      loss_for_classification (logits, labels, is_multilabel=False,
>                               is_multihead=False, head_sizes=[],
>                               head_weights=[])

The general loss function for classification

- If is_multilabel is ```False``` and is_multihead is ```False```: One-Head Classification, e.g. You predict 1 out of n class

- If is_multilabel is ```False``` and is_multihead is ```True```: Multi-Head Classification, e.g. You predict 1 out of n classes at Level 1, 
and 1 out of m classes at Level 2

- If is_multilabel is ```True``` and is_multihead is ```False```: One-Head Multi-Label Classification, e.g. You predict x out of n class (x>=1)

- If is_multilabel is ```True``` and is_multihead is ```True```: Not supported!

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| logits |  |  | output of the last linear layer, before any softmax/sigmoid. Size: (bs,class_size) |
| labels |  |  | determined by your datasetdict. Size: (bs,number_of_head) |
| is_multilabel | bool | False | Whether this is a multilabel classification |
| is_multihead | bool | False | Whether this is a multihead (multi-level) classification |
| head_sizes | list | [] | class size for each head |
| head_weights | list | [] | loss weight for each head |

## Classification head

In [ ]:
#| export
class RobertaConcatHeadExtended(torch.nn.Module):
    """
    Concatenated head for Roberta Classification Model. 

    This head takes the last 4 hidden states (of the last 4 layers), and concatenate them before passing through the classifier head
    """
    def __init__(self,
                 config, # HuggingFace model configuration
                 classifier_dropout=0.1, # Dropout ratio (for dropout layer right before the last nn.Linear)
                 last_hidden_size=768, # Last hidden size (before the last nn.Linear)
                 n_output=None, # Number of label output 
                 **kwargs
                ):

        super().__init__()
        self.last_hidden_size=last_hidden_size
        self.dropout = torch.nn.Dropout(classifier_dropout)
        self.pre_classifier = torch.nn.Linear(4*config.hidden_size,last_hidden_size)
        self.out_proj = torch.nn.Linear(last_hidden_size, n_output)
    
    def forward(self, inp, **kwargs):
        x = inp
        x = self.dropout(x)
        x = self.pre_classifier(x)
        x = torch.tanh(x)
#         x = torch.relu(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x

In [ ]:
show_doc(RobertaConcatHeadExtended)

---

[source](https://github.com/anhquan0412/that-nlp-library/blob/main/that_nlp_library/models/classifiers.py#L132){target="_blank" style="float:right; font-size:smaller"}

### RobertaConcatHeadExtended

>      RobertaConcatHeadExtended (config, classifier_dropout=0.1,
>                                 last_hidden_size=768, n_output=None, **kwargs)

Concatenated head for Roberta Classification Model. 

This head takes the last 4 hidden states (of the last 4 layers), and concatenate them before passing through the classifier head

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| config |  |  | HuggingFace model configuration |
| classifier_dropout | float | 0.1 | Dropout ratio (for dropout layer right before the last nn.Linear) |
| last_hidden_size | int | 768 | Last hidden size (before the last nn.Linear) |
| n_output | NoneType | None | Number of label output |
| kwargs |  |  |  |

In [ ]:
#| export
class RobertaConcatHeadSimple(torch.nn.Module):
    """
    Concatenated head for Roberta Classification Model, the simpler version (no hidden linear layer)

    This head takes the last 4 hidden states (of the last 4 layers), and concatenate them before passing through the classifier head
    """
    def __init__(self,
                 config, # HuggingFace model configuration
                 classifier_dropout=0.1, # Dropout ratio (for dropout layer right before the last nn.Linear)
                 n_output=None, # Number of label output
                 **kwargs
                ):

        super().__init__()
        self.dropout = torch.nn.Dropout(classifier_dropout)
        self.out_proj = torch.nn.Linear(4*config.hidden_size, n_output)
    def forward(self, inp, **kwargs):
        x = inp
        x = self.dropout(x)
        x = self.out_proj(x)
        return x

In [ ]:
show_doc(RobertaConcatHeadSimple)

---

[source](https://github.com/anhquan0412/that-nlp-library/blob/main/that_nlp_library/models/classifiers.py#L163){target="_blank" style="float:right; font-size:smaller"}

### RobertaConcatHeadSimple

>      RobertaConcatHeadSimple (config, classifier_dropout=0.1, n_output=None,
>                               **kwargs)

Concatenated head for Roberta Classification Model, the simpler version (no hidden linear layer)

This head takes the last 4 hidden states (of the last 4 layers), and concatenate them before passing through the classifier head

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| config |  |  | HuggingFace model configuration |
| classifier_dropout | float | 0.1 | Dropout ratio (for dropout layer right before the last nn.Linear) |
| n_output | NoneType | None | Number of label output |
| kwargs |  |  |  |

In [ ]:
#| export
class RobertaClassificationHeadCustom(torch.nn.Module):
    """
    Same as RobertaClassificationHead, but you can freely adjust dropout
    
    Reference: https://github.com/huggingface/transformers/blob/dbc12269ed5546b2da9236b9f1078b95b6a4d3d5/src/transformers/models/roberta/modeling_roberta.py#L1449
    """
    
    def __init__(self, 
                 config, # HuggingFace model configuration
                 classifier_dropout=0.1, # Dropout ratio (for dropout layer right before the last nn.Linear)
                 n_output=None # Number of label output
                ):
        super().__init__()
        self.dense = torch.nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = torch.nn.Dropout(classifier_dropout)
        self.out_proj = torch.nn.Linear(config.hidden_size, n_output)

    def forward(self, features, **kwargs):
        x = features[:, 0, :]  # take <s> token (equiv. to [CLS])
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x

In [ ]:
show_doc(RobertaClassificationHeadCustom)

---

[source](https://github.com/anhquan0412/that-nlp-library/blob/main/that_nlp_library/models/classifiers.py#L186){target="_blank" style="float:right; font-size:smaller"}

### RobertaClassificationHeadCustom

>      RobertaClassificationHeadCustom (config, classifier_dropout=0.1,
>                                       n_output=None)

Same as RobertaClassificationHead, but you can freely adjust dropout

Reference: https://github.com/huggingface/transformers/blob/dbc12269ed5546b2da9236b9f1078b95b6a4d3d5/src/transformers/models/roberta/modeling_roberta.py#L1449

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| config |  |  | HuggingFace model configuration |
| classifier_dropout | float | 0.1 | Dropout ratio (for dropout layer right before the last nn.Linear) |
| n_output | NoneType | None | Number of label output |

## Main classification architecture

In [ ]:
#| export
class RobertaBaseForSequenceClassification(RobertaPreTrainedModel):
    """
    Base Roberta Architecture for Sequence Classification task
    """
    # make sure standard XLM-R are used
    config_class = RobertaConfig

    def __init__(self,
                 config, # HuggingFace model configuration
                 pretrained_roberta=None, # HuggingFace Roberta Body (useful for knowledge transfering task)
                 classifier_dropout=0.1, # Dropout ratio (for dropout layer right before the last nn.Linear)
                 is_multilabel=False, # Whether this is a multilabel classification
                 is_multihead=False, # Whether this is a multihead (multi-level) classification
                 head_class_sizes=[], # Class size for each head
                 head_weights=[], # loss weight for each head. This will be multiplied to the loss of each head's output
                ):
        super().__init__(config)
        self.is_multilabel = is_multilabel
        self.is_multihead = is_multihead
        self.head_class_sizes = val2iterable(head_class_sizes)
        self.head_weights = val2iterable(head_weights,lsize=len(self.head_class_sizes))
        
        # set num_labels for config
        num_labels = sum(self.head_class_sizes)
        config.num_labels = num_labels
        
        # add_pooling_layer to False to ensure all hidden states are returned and not only the one associated with the [CLS] token.
        self.roberta = RobertaModel(config, add_pooling_layer=False) if pretrained_roberta is None else pretrained_roberta
        # Set up token classification head
        self.classification_head = RobertaClassificationHeadCustom(config=config,
                                                                   classifier_dropout=classifier_dropout,
                                                                   n_output=num_labels) 

        # Load and initialize weights from RobertaPretrainedModel
        if pretrained_roberta is None:
            self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                labels=None, **kwargs):
        # Use model body to get encoder representations
        # the only ones we need for now are input_ids and attention_mask
        outputs = self.roberta(input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids, **kwargs)
        
        sequence_output = outputs[0]
        logits = self.classification_head(sequence_output) # (bs,sum of all class sizes)
        
        
        # Calculate losses
        if labels is None:
            loss=None
        else:
            loss = loss_for_classification(logits, labels, 
                                       self.is_multilabel,
                                       self.is_multihead, 
                                       self.head_class_sizes,
                                       self.head_weights)
            

        # Return model output object
        return SequenceClassifierOutput(loss=loss, logits=logits,
#                                      hidden_states=outputs.hidden_states,
                                        hidden_state=None,
                                     attentions=outputs.attentions)

In [ ]:
show_doc(RobertaBaseForSequenceClassification)

---

[source](https://github.com/anhquan0412/that-nlp-library/blob/main/that_nlp_library/models/classifiers.py#L213){target="_blank" style="float:right; font-size:smaller"}

### RobertaBaseForSequenceClassification

>      RobertaBaseForSequenceClassification (config, pretrained_roberta=None,
>                                            classifier_dropout=0.1,
>                                            is_multilabel=False,
>                                            is_multihead=False,
>                                            head_class_sizes=[],
>                                            head_weights=[])

Base Roberta Architecture for Sequence Classification task

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| config |  |  | HuggingFace model configuration |
| pretrained_roberta | NoneType | None | HuggingFace Roberta Body (useful for knowledge transfering task) |
| classifier_dropout | float | 0.1 | Dropout ratio (for dropout layer right before the last nn.Linear) |
| is_multilabel | bool | False | Whether this is a multilabel classification |
| is_multihead | bool | False | Whether this is a multihead (multi-level) classification |
| head_class_sizes | list | [] | Class size for each head |
| head_weights | list | [] | loss weight for each head. This will be multiplied to the loss of each head's output |

In [ ]:
#| export
class RobertaHiddenStateConcatForSequenceClassification(RobertaPreTrainedModel):
    """
    Roberta Architecture with Hidden-State-Concatenation for Sequence Classification task
    """
    
    config_class = RobertaConfig

    def __init__(self,config, # HuggingFace model configuration
                 pretrained_roberta=None, # HuggingFace Roberta Body (useful for knowledge transfering task)
                 concathead_class=RobertaConcatHeadSimple, # Head object, e.g. RobertaConcatHeadSimple
                 classifier_dropout=0.1, # Dropout ratio (for dropout layer right before the last nn.Linear)
                 last_hidden_size=768, # Last hidden size (before the last nn.Linear)
                 is_multilabel=False, # Whether this is a multilabel classification
                 is_multihead=False, # Whether this is a multihead (multi-level) classification
                 head_class_sizes=[], # Class size for each head
                 head_weights=[], # loss weight for each head. This will be multiplied to the loss of each head's output
                ):
        super().__init__(config)
        self.is_multilabel = is_multilabel
        self.is_multihead = is_multihead
        self.head_class_sizes = val2iterable(head_class_sizes)
        self.head_weights = val2iterable(head_weights,lsize=len(self.head_class_sizes))
        
        # set num_labels for config
        num_labels = sum(self.head_class_sizes)
        config.num_labels = num_labels
        
        # Load model body
        # add_pooling_layer to False to ensure all hidden states are returned  and not only the one associated with the [CLS] token.
        self.roberta = RobertaModel(config, add_pooling_layer=False) if pretrained_roberta is None else pretrained_roberta
        
        # Set up token classification head
        self.classification_head = concathead_class(config=config,
                                                    classifier_dropout=classifier_dropout,
                                                    last_hidden_size=last_hidden_size,
                                                   n_output=num_labels) 

        # Load and initialize weights from RobertaPretrainedModel
        if pretrained_roberta is None:
            self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                labels=None, **kwargs):
        # Use model body to get encoder representations
        # the only ones we need for now are input_ids and attention_mask
        outputs = self.roberta(input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids, **kwargs)
        
        hidden_states = outputs['hidden_states'] # tuples with len 13 (number of layer/block)
        # each with shape: (bs,seq_len,hidden_size_len), e.g. for phobert: (bs,256, 768)
        # Note: hidden_size_len = embedding_size
        hidden_concat = torch.cat([hidden_states[i][:,0] for i in range(-1,-4-1,-1)],
                                  -1) 
        logits = self.classification_head(hidden_concat) # (bs,sum of all class sizes)
        # Calculate losses
        
        if labels is None:
            loss=None
        else:
            loss = loss_for_classification(logits, labels, 
                                       self.is_multilabel,
                                       self.is_multihead, 
                                       self.head_class_sizes,
                                       self.head_weights)

            
        # Return model output object
        return SequenceClassifierOutput(loss=loss, logits=logits,
#                                      hidden_states=outputs.hidden_states,
                                     hidden_states=None,                                        
                                     attentions=outputs.attentions)

In [ ]:
show_doc(RobertaHiddenStateConcatForSequenceClassification)

---

[source](https://github.com/anhquan0412/that-nlp-library/blob/main/that_nlp_library/models/classifiers.py#L279){target="_blank" style="float:right; font-size:smaller"}

### RobertaHiddenStateConcatForSequenceClassification

>      RobertaHiddenStateConcatForSequenceClassification (config,
>                                                         pretrained_roberta=Non
>                                                         e, concathead_class=<c
>                                                         lass '__main__.Roberta
>                                                         ConcatHeadSimple'>, cl
>                                                         assifier_dropout=0.1,
>                                                         last_hidden_size=768,
>                                                         is_multilabel=False,
>                                                         is_multihead=False,
>                                                         head_class_sizes=[],
>                                                         head_weights=[])

Roberta Architecture with Hidden-State-Concatenation for Sequence Classification task

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| config |  |  | HuggingFace model configuration |
| pretrained_roberta | NoneType | None | HuggingFace Roberta Body (useful for knowledge transfering task) |
| concathead_class | type | RobertaConcatHeadSimple | Head object, e.g. RobertaConcatHeadSimple |
| classifier_dropout | float | 0.1 | Dropout ratio (for dropout layer right before the last nn.Linear) |
| last_hidden_size | int | 768 | Last hidden size (before the last nn.Linear) |
| is_multilabel | bool | False | Whether this is a multilabel classification |
| is_multihead | bool | False | Whether this is a multihead (multi-level) classification |
| head_class_sizes | list | [] | Class size for each head |
| head_weights | list | [] | loss weight for each head. This will be multiplied to the loss of each head's output |

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()